# Linear buckling and prestressed natural frequencies of a 3D column

This tutorial demonstrates two eigenvalue analyses of a slender, clamped cantilever modeled with three-dimensional solid finite elements:

1. **Linear buckling:** determine the load multipliers at which the equilibrium loses stability.
2. **Prestressed modal analysis:** determine how the same compressive reference load changes the natural frequencies.

The linear elastic stiffness, geometric stiffness and mass matrices are all assembled directly from **weak forms** in LowLevelFEM. No dedicated element-level buckling formulation is needed.

The column is aligned with the global $x$ axis, has length $L=100$ and a square $1\times1$ cross-section, and is fixed at $x=0$. A compressive reference force $P_0=1$ acts on the opposite end. The example uses consistent N–mm–tonne–s units.

## 1. Geometry, material and displacement field

`structured_box_mesh` creates a structured hexahedral mesh with physical groups `"body"`, `"left"` and `"right"`. The subdivision parameter controls both cross-sectional and axial resolution; because the beam is slender, the total number of degrees of freedom grows rapidly.

The default below uses `n=4` so the entire notebook remains practical to run. Set `n=10` to reproduce the fine-mesh study (approximately 363,000 degrees of freedom) and obtain closer agreement with Euler's beam formula.

We use $E=200\,000$, $\nu=0.3$ and $\rho=7.85\times10^{-9}$ (consistent with the units above). The unknown $\mathbf u$ is a three-component displacement field.

In [15]:
using LowLevelFEM, LinearAlgebra, Arpack

In [16]:
L = 100.0
b = 1.0
h = 1.0
n = 4  # Change to 10 for the fine-mesh reference model.
structured_box_mesh(lx=L, ly=b, lz=h, n=n)
# openPreProcessor()

In [17]:
mat = Material("body", E=2e5, ν=0.3, ρ=7.85e-9)
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

## 2. Linear elastic equilibrium under the reference load

For the infinitesimal-strain, linear elastic problem, the weak form is

$$
\int_\Omega \boldsymbol\varepsilon(\mathbf v):\mathbb C:\boldsymbol\varepsilon(\mathbf u_0)\,\mathrm d\Omega
= \int_{\Gamma_R}\mathbf v\cdot\bar{\mathbf t}\,\mathrm d\Gamma,
$$

where $\boldsymbol\varepsilon(\mathbf u)=\tfrac12(\nabla\mathbf u+\nabla\mathbf u^T)$ and $\bar{\mathbf t}=(-P_0,0,0)$ denotes a uniform end traction. Since the loaded face has unit area, its total resultant force has magnitude $P_0=1$.

`SymGrad(U)` evaluates engineering strains (the shear components are twice the tensorial shear strains), while `D(:Solid, mat)` supplies the corresponding elastic material matrix. All displacements are fixed on the `"left"` face.

In [18]:
K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
P0 = 1.0
f = ∫(U ⋅ [-P0, 0.0, 0.0], Γ="right")
bc = BoundaryCondition("left", ux=0, uy=0, uz=0);

In [19]:
u0 = solveField(Symmetric(K), f; support=[bc])
showDoFResults(u0; name="Static compression", factor=1);

## 3. Pre-stress field

The compressive load generates the reference stress field $\boldsymbol\sigma_0$. With isotropic Hooke's law,

$$
\boldsymbol\sigma_0
=\frac{E}{1+\nu}\left[
\boldsymbol\varepsilon(\mathbf u_0)
+\frac{\nu}{1-2\nu}\operatorname{tr}(\boldsymbol\varepsilon(\mathbf u_0))\,\mathbf I
\right].
$$

The strain is evaluated as a finite element tensor field. Near the clamped end the stress is not exactly uniaxial; away from it, $\sigma_{xx}$ is close to $-P_0/(bh)=-1$.

Here we use small-strain linear elasticity and a reference-configuration geometric stiffness. This is a **linearized pre-stress approximation**, not a geometrically nonlinear equilibrium analysis.

In [20]:
ε = (u0 ∘ ∇ + ∇ ∘ u0) / 2
E = mat.E
ν = mat.ν
I3 = unitTensor(ε)
σ0 = E / (1 + ν) * (ε + ν / (1 - 2ν) * trace(ε) * I3)
s = tensorToMatrix(σ0);

## 4. Geometric stiffness from the weak form

The initial-stress (geometric) stiffness is defined by

$$
\mathbf v^{T} K_G\mathbf u
=\int_\Omega\sum_{i=1}^{3}
(\nabla v_i)^T\boldsymbol\sigma_0(\nabla u_i)\,\mathrm d\Omega.
$$

In LowLevelFEM, `Grad(U)` groups the nine displacement-gradient components by displacement component. Consequently, the coefficient for the bilinear form is the following $9\times9$ block matrix:

$$
D_G=\begin{bmatrix}
\boldsymbol\sigma_0&0&0\\
0&\boldsymbol\sigma_0&0\\
0&0&\boldsymbol\sigma_0
\end{bmatrix}.
$$

A `Matrix{Any}` can contain both numeric zeroes and the spatially varying `ScalarField` components of `s`. Compression typically produces negative contributions to $K_G$; we **do not** change that sign during assembly.

In [21]:
DG = Matrix{Any}(zeros(9, 9))

DG[1:3, 1:3] = s
DG[4:6, 4:6] = s
DG[7:9, 7:9] = s

Kg = ∫(Grad(U) ⋅ DG ⋅ Grad(U));

## 5. Linear buckling eigenvalue problem

Let $P=\alpha P_0$ be an applied load proportional to the reference load. The linearized stability equation is

$$
(K+\alpha K_G)\boldsymbol\phi=0.
$$

The generalized eigenvalue problem solved below is

$$
K_G\boldsymbol\phi=\mu K\boldsymbol\phi,
\qquad \boxed{\alpha_{\mathrm{cr}}=-\frac{1}{\mu}}.
$$

For compressive stress, the relevant $\mu$ are negative. `solveEigen` returns the mode shapes in a `VectorField` and stores the eigenvalues in its `t` vector. Rather than changing `t` in place, `mapEigenValues` returns a field with positive critical load multipliers while keeping each eigenvector paired with its eigenvalue.

For a square cross-section, the first bending modes are expected to be nearly degenerate, because buckling can occur in either transverse direction.

In [22]:
buckling_modes = solveEigen(Kg, K; support=[bc], nev=6, which=:SR)
buckling_modes = mapEigenValues(μ -> -1 / μ, buckling_modes)
critical_factors = getEigenValues(buckling_modes)
critical_factors

6-element Vector{Float64}:
   4.253680602476314
   4.253680701916678
  38.259258941054426
  38.25925904049526
 106.14342833134805
 106.14342838076585

### Euler beam-theory check

For a cantilever with effective length $2L$, the lowest Euler critical force is

$$
P_\mathrm{Euler}=\frac{\pi^2 E I}{4 L^2},\qquad
I=\frac{bh^3}{12}=\frac{hb^3}{12}.
$$

With this geometry, $P_\mathrm{Euler}\approx 4.1123$. Since the reference resultant is $P_0=1$, the first critical *load factor* should also be close to $4.1123$. The 3D continuum model may give a slightly different value, especially on a coarser mesh.

**Interpretation:** the first critical force is $P_{\mathrm{cr}}=\alpha_{\mathrm{cr}}P_0$. Linearized buckling predicts the *onset* of instability, not the post-buckling displacement amplitude.

In [23]:
Imin = min(b * h^3 / 12, h * b^3 / 12)
P_euler = π^2 * E * Imin / (4 * L^2)
Pcr_fem = first(critical_factors) * P0

println("Euler critical force: ", round(P_euler; digits=6))
println("FEM critical force:   ", round(Pcr_fem; digits=6))
println("Relative difference:   ", round(100 * (Pcr_fem / P_euler - 1); digits=3), " %")

Euler critical force: 4.112335
FEM critical force:   4.253681
Relative difference:   3.437 %


In [24]:
showEigenResults(buckling_modes; name="Buckling modes", ff=2, factor=500);

## 6. Natural frequencies with and without pre-stress

The **consistent mass matrix** follows from the kinetic-energy bilinear form:

$$
\mathbf v^T M\mathbf u
=\int_\Omega \rho\,\mathbf v\cdot\mathbf u\,\mathrm d\Omega.
$$

Without pre-stress, the undamped modal eigenproblem is

$$
K\boldsymbol\phi=\omega_0^2 M\boldsymbol\phi.
$$

With a compressive load $P=\alpha P_0$, the geometric stiffness from above modifies the tangent stiffness:

$$
\boxed{(K+\alpha K_G)\boldsymbol\phi=\omega_\alpha^2M\boldsymbol\phi}.
$$

In this example $\alpha=1$, meaning that the **same** reference load used in the static analysis is present during the vibration analysis. Frequencies in hertz are computed as $f=\omega/(2\pi)=\sqrt{\omega^2}/(2\pi)$.

Because compression softens the lateral response, the lowest natural frequencies should decrease. This linearized formulation is valid while the prestressed state remains stable, i.e. below the first critical load factor.

In [25]:
M = ∫(U ⋅ mat.ρ ⋅ U)

# The default model uses a compressive reference load factor α = 1.
α = 1.0
Kprestressed = K + α * Kg

free_modes = solveEigen(Symmetric(K), Symmetric(M);
    support=[bc], nev=6, shift=0.0)
prestressed_modes = solveEigen(Symmetric(Kprestressed), Symmetric(M);
    support=[bc], nev=6, shift=0.0);

### Compare the natural frequencies

In both modal analyses, the generalized eigenvalues represent $\omega^2$, **not** $\omega$ and not frequencies in hertz. We check that the requested eigenvalues are positive before converting them.

The square cross-section gives nearly paired bending modes. Their numerical ordering can interchange between the unloaded and prestressed calculations; compare frequency *groups*, not individual mode directions, when the frequencies are almost equal.

In [26]:
ω2_free = getEigenValues(free_modes)
ω2_pre = getEigenValues(prestressed_modes)

all(x -> x > 0.0, ω2_free) && all(x -> x > 0.0, ω2_pre) ||
    error("Non-positive modal eigenvalue: the selected state may be unstable.")

freq_free = sqrt.(ω2_free) ./ (2π)
freq_pre = sqrt.(ω2_pre) ./ (2π)

println("Mode     Unloaded [Hz]   Prestressed [Hz]   Change [%]")
for i in eachindex(freq_pre)
    change = 100 * (freq_pre[i] / freq_free[i] - 1)
    println(lpad(i, 4), "     ",
        lpad(round(freq_free[i]; digits=4), 12), "     ",
        lpad(round(freq_pre[i]; digits=4), 15), "     ",
        lpad(round(change; digits=2), 10))
end

Mode     Unloaded [Hz]   Prestressed [Hz]   Change [%]
   1          82.9521             73.1876         -11.77
   2          82.9521             73.1876         -11.77
   3           519.62            509.4409          -1.96
   4           519.62            509.4409          -1.96
   5        1453.9193           1445.3177          -0.59
   6        1453.9193           1445.3177          -0.59


### Visualize the prestressed vibration modes

For visualization we can also transform the modal eigenvalues $\omega^2$ into frequencies $f$ using `mapEigenValues`. This does not modify the eigenvectors or the original `prestressed_modes` field. The resulting Gmsh view labels the modes by their frequencies rather than by squared angular frequencies.

The eigenvectors have arbitrary amplitude (and sign); the displayed deformation factor changes the visualization only, not the natural frequencies.

In [27]:
frequency_modes = LowLevelFEM.mapEigenValues(ω2 -> sqrt(ω2) / (2π), prestressed_modes)
showEigenResults(frequency_modes;
    name="Prestressed natural frequencies [Hz]", ff=1, factor=500);

## 7. Summary and limitations

- The **same geometric stiffness** $K_G$ is used for both eigenproblems.
- Buckling uses $(K+\alpha K_G)\boldsymbol\phi=0$ to find the critical load factor $\alpha$.
- Prestressed vibration uses $(K+\alpha K_G)\boldsymbol\phi=\omega^2M\boldsymbol\phi$ at a specified pre-load $\alpha$.
- In the elastic, stable range, increasing a compressive load decreases the lowest lateral natural frequencies. As the load approaches the critical factor, the first corresponding squared frequency approaches zero.
- The analysis assumes **linear elasticity, small strains and a linearly scaled initial stress field**. Large-deformation and post-buckling effects require a different, consistent tangent formulation.

Use the Gmsh post-processor to switch between the buckling and vibration views and to inspect individual modes.

In [28]:
openPostProcessor();